# Business Entity Resolution Challenge — Google Colab Runner

This notebook automates the complete pipeline execution on Google Colab:
1. **Mount Google Drive & Unzip Student Resource**
2. **Clone & Setup Pipeline Code**
3. **Install Dependencies**
4. **Train Model, Tune F0.5 Threshold, & Generate Test Matches**
5. **Validate Submission Files**
6. **Package Final Submission ZIP**

### Step 1: Mount Google Drive & Extract Student Resource Data
Ensure `student_resource.zip` is placed in your Google Drive (e.g., `MyDrive/6ab10eb3b23ba_student_resource.zip`).

In [ ]:
from google.colab import drive
import os
import glob

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define zip location in your Drive
zip_path = '/content/drive/MyDrive/6ab10eb3b23ba_student_resource.zip'
extract_path = '/content/student_resource'

# Auto-detect if exact filename differs slightly
if not os.path.exists(zip_path):
    matches = glob.glob('/content/drive/MyDrive/*student_resource*.zip')
    if matches:
        zip_path = matches[0]
        print(f"Found student resource zip at: {zip_path}")

# 3. Unzip the file silently into Colab's fast local runtime
if os.path.exists(zip_path):
    print("Zip file found. Unzipping data... Please wait.")
    !unzip -q "{zip_path}" -d "{extract_path}"
    print(f"Done! Files extracted to: {extract_path}")
    # Print directory contents to verify structure
    !ls -l "{extract_path}"
else:
    print(f"❌ Error: Could not find the file at {zip_path}. Please check if the file is in your Google Drive.")

### Step 2: Clone or Pull Latest Pipeline Code from GitHub

In [ ]:
# Clone the ML pipeline repository
%cd /content/
if not os.path.exists('/content/ML-Challenge-Training-pipeline'):
    !git clone https://github.com/Aayush349/ML-Challenge-Training-pipeline.git
else:
    %cd /content/ML-Challenge-Training-pipeline
    !git pull

# Copy code folder into student_resource directory
!mkdir -p /content/student_resource/code
!cp -r /content/ML-Challenge-Training-pipeline/code/business_entity_resolution /content/student_resource/code/
!cp -r /content/ML-Challenge-Training-pipeline/utils /content/student_resource/ 2>/dev/null || true
print("✅ Pipeline code successfully set up in /content/student_resource")

### Step 3: Install Required Dependencies

In [ ]:
!pip install -q rapidfuzz faiss-cpu sentence-transformers lightgbm

### Step 4: Run End-to-End Pipeline
Executes preprocessing, TF-IDF + MiniLM FAISS candidate blocking, LightGBM model training with PR-AUC early stopping, F0.5 threshold calibration, and test set inference.

In [ ]:
%cd /content/student_resource
!python code/business_entity_resolution/src/run_pipeline.py

### Step 5: Validate Output Files (Pre-Submission Sanity Check)

In [ ]:
%cd /content/student_resource
!python utils/validate_submission.py \
  --matching output/matching_results.tsv \
  --candidate output/candidate_pairs.tsv \
  --test-dir dataset/test

### Step 6: Download Leaderboard TSV & Package Final Submission Zip

In [ ]:
%cd /content/student_resource

# Package full zip according to competition guidelines
!zip -r /content/final_submission_package.zip \
  output/matching_results.tsv \
  output/candidate_pairs.tsv \
  code/business_entity_resolution/ \
  Documentation_template.md 2>/dev/null || zip -r /content/final_submission_package.zip output/ code/

from google.colab import files
print("Downloading matching_results.tsv for leaderboard upload...")
files.download('output/matching_results.tsv')

print("Downloading final submission archive...")
files.download('/content/final_submission_package.zip')